In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from topoflow.flow.paths import GaussianConditionalProbabilityPath
from topoflow.flow.schedulers import LinearAlpha, LinearBeta
from topoflow.utils.visualization import visualize_data_rdkit, visualize_data, visualize_trajectory, visualize_data_matplotlib
from topoflow.flow.flow_models_ligand import FlowModel
from topoflow.data.dataloader_adj import DataLoaderSampleable
from topoflow.data.datasets import CrossDocked
from topoflow.flow.schedulers import LinearAlpha, LinearBeta, SineAlpha, CosineBeta

In [3]:
batch_size = 32
data_dir = "../data/CrossDocked"
train_dataset = CrossDocked(
    root=data_dir,
    split="train",
)
val_dataset = CrossDocked(root=data_dir, split="val")

alpha = SineAlpha()
beta = CosineBeta()

p_data = DataLoaderSampleable(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
)


distribution_hist = np.load("../data/CrossDocked/processed/size_distribution_full.npy", allow_pickle=True)
path = GaussianConditionalProbabilityPath(p_data=p_data, alpha=alpha, beta=beta, distribution_hist=distribution_hist)
checkpoint_path = "../checkpoints/runs_cluster/30_09_154e_ast_last_step2.ckpt"

#Load the model from the checkpoint
model = FlowModel.load_from_checkpoint(checkpoint_path, path=path, norm_path="../data/CrossDocked/processed/normalization_params_full.npz")

Loaded dataset from ../data/CrossDocked/processed/train_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/train_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 946 samples.
Loaded dataset from ../data/CrossDocked/processed/val_crossdocked_pyg.pt.
Loaded SMILES file from ../data/CrossDocked/processed/val_crossdocked_pyg_smiles.txt
Dataset loaded successfully with 244 samples.


/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Entropy of n_nodes: H[N] 9.08344554901123


/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'vector_field_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['vector_field_model'])`.
/home/marco/miniforge3/envs/topoflow/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'time_factor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['time_factor'])`.


In [4]:
res_traj = model.sample_with_trajectory(1, n_steps=100, force_connected=False)
visualize_trajectory(res_traj, image_path="outputs/trajectory/mol_traj.png", gif_path="outputs/trajectory/mol_traj.gif", plot_every=2)

Visualizing trajectory:  63%|██████▎   | 63/100 [00:27<00:14,  2.48it/s][09:16:11] Both bonds on one end of an atropisomer are on the same side - atoms are: 20 15
[09:16:11] Both bonds on one end of an atropisomer are on the same side - atoms are: 0 27
[09:16:11] Both bonds on one end of an atropisomer are on the same side - atoms are: 13 7
Visualizing trajectory:  65%|██████▌   | 65/100 [00:28<00:14,  2.45it/s][09:16:12] Both bonds on one end of an atropisomer are on the same side - atoms are: 5 20
[09:16:12] Both bonds on one end of an atropisomer are on the same side - atoms are: 0 27
[09:16:12] Both bonds on one end of an atropisomer are on the same side - atoms are: 13 7
[09:16:12] The 2 defining bonds for an atropisomer are co-planar - atoms are: 8 20
Visualizing trajectory:  67%|██████▋   | 67/100 [00:29<00:13,  2.42it/s][09:16:12] The 2 defining bonds for an atropisomer are co-planar - atoms are: 8 20
[09:16:12] Both bonds on one end of an atropisomer are on the same side - ato

In [ ]:
res = model.sample(1, n_steps=20)

Seed set to 42


In [9]:
visualize_data(res, image_path="outputs/generated_molecules_pymol.png", session_path="outputs/generated_molecules_session.pse")

In [8]:
visualize_data_rdkit(res, image_path="outputs/generated_molecules.png", mols_per_row=2, sub_img_size=(1200, 1200))

Image saved to: outputs/generated_molecules.png


[16:46:19] Both bonds on one end of an atropisomer are on the same side - atoms are: 17 25


In [47]:
distances, other_metrics = model.test_distribution(
                test_distribution=model.val_distribution,
                metrics=[
                    "qed",
                    "logp",
                    "sas",
                ],  # TODO: add lipinski, but also in the val_dist
                atom_decoder=model.atom_decoder,
            )

Sampling molecules: 100%|██████████████████████████████████████████| 2/2 [00:20<00:00, 10.33s/batch]


In [49]:
model.normalization_params

{'mean': array([ 7.7809190e-08, -9.0558565e-08,  5.2221260e-08], dtype=float32),
 'std': array([5.8403964, 5.7269087, 5.8340626], dtype=float32)}

In [48]:
other_metrics

{'validity': tensor(1.), 'connectivity': tensor(0.), 'uniqueness': tensor(0.)}

In [ ]:
from torch_geometric.data import Data
from topoflow.utils.metrics import compute_connectivity
from topoflow.utils.molecules import data_to_rdmol

final_mol = res_traj[-1]
final_rdmol = data_to_rdmol(final_mol, atom_decoder=model.atom_decoder, sanitize=False)

connected = compute_connectivity(final_rdmol)[0]
print(connected)

[]


In [ ]:
res_traj = model._test_integration(n_steps=80)
visualize_trajectory(res_traj, image_path="outputs/trajectory/mol_traj.png", gif_path="outputs/trajectory/mol_traj.gif", plot_every=2)